# Penggabungan Dataset BNaT

Notebook singkat ini menyatukan seluruh berkas CSV dari folder `BNaT`
menjadi satu DataFrame lengkap dengan struktur kolom yang mengikuti
`docs/extractor_headers.txt`. Jalankan cell secara berurutan untuk memuat,
menggabungkan, dan menyimpan hasilnya.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import pandas as pd


def resolve_project_root() -> Path:
    """Cari root proyek dengan mencari folder BNaT di calon direktori."""
    candidate_paths: list[Path] = []
    try:
        candidate_paths.append(Path(__file__).resolve().parent.parent)
    except NameError:
        pass
    cwd = Path.cwd().resolve()
    candidate_paths.append(cwd)
    candidate_paths.extend(cwd.parents)

    seen: set[Path] = set()
    for candidate in candidate_paths:
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "BNaT").exists():
            return candidate
    return candidate_paths[0]

In [2]:
# Konfigurasi jalur dan parameter ekspor.

PROJECT_ROOT = resolve_project_root()
BNAT_DIR = PROJECT_ROOT / "BNaT"
HEADERS_FILE = BNAT_DIR / "docs" / "extractor_headers.txt"
OUTPUT_TARGET_DIRS = (PROJECT_ROOT / "Dataset",)
EXPORT_BASENAME = "dataset bnat"
EXPORT_FORMATS = ("csv",)  # Tambahkan "parquet" bila pyarrow/fastparquet tersedia.
# Gunakan daftar berikut untuk membatasi file sumber (None berarti seluruh CSV dipakai).
SELECTED_RELATIVE_SOURCES = (
    "Merged_dataset/w1.csv",
    "Merged_dataset/w2.csv",
    "Merged_dataset/w3.csv",
)

In [3]:
# Fungsi utilitas untuk memuat header, mengumpulkan file, dan memproses data.
def load_headers(path: Path) -> list[str]:
    """Parse nama kolom dari file header bawaan BNaT."""
    raw_text = path.read_text(encoding="utf-8").replace("\n", "")
    tokens = [token.strip().strip('"') for token in raw_text.split(",")]
    headers = [token for token in tokens if token]
    if not headers:
        raise ValueError(f"Tidak menemukan header di {path}")
    return headers


def collect_csv_files(
    root_dir: Path, selected_relative: Iterable[str] | None = None
) -> list[Path]:
    """Ambil file CSV sesuai daftar pilihan atau seluruh folder BNaT."""
    if selected_relative:
        csv_files = []
        for rel in selected_relative:
            path = (root_dir / rel).resolve()
            if not path.is_file():
                raise FileNotFoundError(f"File sumber tidak ditemukan: {path}")
            csv_files.append(path)
    else:
        csv_files = sorted(root_dir.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"Tidak ada CSV yang ditemukan di {root_dir}")
    return csv_files

In [4]:
# Fungsi untuk memuat CSV sesuai header asli.
def load_csv(csv_path: Path, headers: list[str]) -> pd.DataFrame:
    """Baca CSV tanpa header dan gunakan nama kolom resmi BNaT."""
    df = pd.read_csv(
        csv_path,
        header=None,
        names=headers,
        low_memory=False,
    )
    return df

In [5]:
# Fungsi untuk menggabungkan beberapa DataFrame.
def concat_csv_files(csv_files: Iterable[Path], headers: list[str]) -> pd.DataFrame:
    """Satukan seluruh DataFrame dari daftar berkas."""
    frames: list[pd.DataFrame] = []
    for csv_path in csv_files:
        df = load_csv(csv_path, headers)
        print(f"Memuat {len(df):>8} baris dari {csv_path.name}")
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

In [6]:
# Fungsi untuk mengekspor DataFrame ke berbagai format.
def export_dataset(
    df: pd.DataFrame, output_dir: Path, base_name: str, formats: Iterable[str]
) -> dict[str, Path]:
    """Simpan dataset gabungan ke format yang dipilih."""
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_paths: dict[str, Path] = {}
    for fmt in formats:
        fmt_lower = fmt.lower()
        target_path = output_dir / f"{base_name}.{fmt_lower}"
        if fmt_lower == "csv":
            df.to_csv(target_path, index=False)
        elif fmt_lower == "parquet":
            df.to_parquet(target_path, index=False)
        else:
            raise ValueError(f"Format {fmt} belum didukung.")
        saved_paths[fmt_lower] = target_path
    return saved_paths

In [7]:
# Proses utama: muat header, kumpulkan file, gabungkan, dan ekspor.
FEATURE_COLUMNS = load_headers(HEADERS_FILE)
CSV_FILES = collect_csv_files(BNAT_DIR, SELECTED_RELATIVE_SOURCES)

print(f"Total kolom fitur: {len(FEATURE_COLUMNS)}")
print(f"Ditemukan {len(CSV_FILES)} file CSV di dalam folder BNaT:")
for csv_path in CSV_FILES:
    print(f"  - {csv_path.relative_to(BNAT_DIR)}")

Total kolom fitur: 22
Ditemukan 3 file CSV di dalam folder BNaT:
  - Merged_dataset\w1.csv
  - Merged_dataset\w2.csv
  - Merged_dataset\w3.csv


In [8]:
# Gabungkan seluruh data dan tampilkan ringkasan.
bnat_df = concat_csv_files(CSV_FILES, FEATURE_COLUMNS)
print(
    f"\nUkuran dataset gabungan: {bnat_df.shape[0]:,} baris x {bnat_df.shape[1]} kolom"
)

if "label" in bnat_df.columns:
    print("\nDistribusi label awal:")
    label_counts = bnat_df["label"].value_counts(dropna=False)
    label_ratio = (label_counts / len(bnat_df)).mul(100).round(2).to_dict()
    for label, count in label_counts.items():
        pct = label_ratio.get(label, 0.0)
        print(f"  - {label:<20s}: {count:>10,} baris ({pct:>5.2f}%)")

    allowed_labels = {"normal", "dos"}
    cleaned_labels = bnat_df["label"].astype(str).str.strip()
    mask_allowed = cleaned_labels.str.lower().isin(allowed_labels)
    filtered_df = bnat_df[mask_allowed].copy()
    removed_rows = len(bnat_df) - len(filtered_df)
    if filtered_df.empty:
        raise ValueError(
            "Tidak ada baris dengan label Normal atau DoS setelah penyaringan."
        )

    print(
        f"\nMenyaring label untuk mempertahankan hanya Normal & DoS "
        f"(menghapus {removed_rows:,} baris)."
    )
    filtered_counts = filtered_df["label"].value_counts(dropna=False)
    filtered_ratio = (filtered_counts / len(filtered_df)).mul(100).round(2).to_dict()
    print("Distribusi label setelah penyaringan:")
    for label, count in filtered_counts.items():
        pct = filtered_ratio.get(label, 0.0)
        print(f"  - {label:<20s}: {count:>10,} baris ({pct:>5.2f}%)")

    bnat_df = filtered_df
    filtered_sample = min(5, len(bnat_df))
    if filtered_sample:
        print("\nContoh baris acak (setelah hanya Normal & DoS):")
        print(bnat_df.sample(filtered_sample, random_state=24).to_string(index=False))
else:
    print("\nKolom 'label' tidak ditemukan di dataset hasil gabungan.")

saved_artifacts: dict[Path, dict[str, Path]] = {}
for output_dir in OUTPUT_TARGET_DIRS:
    saved_artifacts[output_dir] = export_dataset(
        bnat_df,
        output_dir,
        EXPORT_BASENAME,
        EXPORT_FORMATS,
    )

print("\nDataset tersimpan:")
for directory, artifacts in saved_artifacts.items():
    print(f"  Folder: {directory}")
    for fmt, path in artifacts.items():
        print(f"    - {fmt.upper():7s}: {path}")

Memuat    70000 baris dari w1.csv
Memuat    70000 baris dari w2.csv
Memuat    70000 baris dari w3.csv

Ukuran dataset gabungan: 210,000 baris x 22 kolom

Distribusi label awal:
  - Normal              :    150,000 baris (71.43%)
  - MitM                :     15,000 baris ( 7.14%)
  - BP                  :     15,000 baris ( 7.14%)
  - DoS                 :     15,000 baris ( 7.14%)
  - FoT                 :     15,000 baris ( 7.14%)

Menyaring label untuk mempertahankan hanya Normal & DoS (menghapus 45,000 baris).
Distribusi label setelah penyaringan:
  - Normal              :    150,000 baris (90.91%)
  - DoS                 :     15,000 baris ( 9.09%)

Contoh baris acak (setelah hanya Normal & DoS):
 duration protocol_type service  src_bytes  dst_bytes flag  count  srv_count  serror_rate  same_srv_rate  diff_srv_rate  srv_serror_rate  srv_diff_host_rate  dst_host_count  dst_host_srv_count  dst_host_same_srv_rate  dst_host_diff_srv_rate  dst_host_same_src_port_rate  dst_host_serror_ra